# Single-Pair Mass Posterior

This example reads the first observed orbital separation in `logD.csv` and evaluates the repository's existing pairwise prior, orbital likelihood, and combined log posterior over a finite grid in $(\log_{10}\tilde{\mu},\gamma)$. The contour panels show relative prior, likelihood, and posterior values. They are deterministic evaluations of the model used by `MCMC_single_pair.py`.

The grid is not a sampled chain. It does not estimate Monte Carlo uncertainty or establish convergence. The prior and logistic orbital-stability likelihood retain the assumptions encoded in `pairwise_model.py`; the figure is not an occurrence-rate calculation or a stability result for an unmodeled system.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    candidate for candidate in search_roots
    if (candidate / "pairwise_model.py").is_file()
    and (candidate / "logD.csv").is_file()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from pairwise_model import combine_log_posterior, log_prior, orbital_log_likelihood
from tdpy.verbosity import print

catalog_path = repository_root / "logD.csv"
print("Reading from logD.csv...")
catalog = pd.read_csv(catalog_path)
observed_log_separation = float(catalog.iloc[0, 0])
print(f"Using the first of {len(catalog)} recorded pair separations.")

In [ ]:
def single_pair_log_likelihood(theta):
    return orbital_log_likelihood(theta[0], observed_log_separation)


log_mu_grid = np.linspace(-6.0, -2.0, 121)
gamma_grid = np.linspace(0.01, 0.99, 99)
log_prior_grid = np.empty((gamma_grid.size, log_mu_grid.size))
log_likelihood_grid = np.empty_like(log_prior_grid)
log_posterior_grid = np.empty_like(log_prior_grid)

for row, gamma in enumerate(gamma_grid):
    for column, log_mu_tilde in enumerate(log_mu_grid):
        theta = (float(log_mu_tilde), float(gamma))
        log_prior_grid[row, column] = log_prior(theta)
        log_likelihood_grid[row, column] = single_pair_log_likelihood(theta)
        log_posterior_grid[row, column] = combine_log_posterior(
            theta, log_prior, single_pair_log_likelihood
        )

relative_prior = np.exp(log_prior_grid - np.max(log_prior_grid))
relative_likelihood = np.exp(log_likelihood_grid - np.max(log_likelihood_grid))
relative_posterior = np.exp(log_posterior_grid - np.max(log_posterior_grid))
posterior_peak = np.unravel_index(np.argmax(log_posterior_grid), log_posterior_grid.shape)
print(
    "Grid posterior maximum at "
    f"log10(mu_tilde)={log_mu_grid[posterior_peak[1]]:.2f}, "
    f"gamma={gamma_grid[posterior_peak[0]]:.2f}."
)

In [ ]:
typefileplot = "png"
if typefileplot not in {"png", "pdf"}:
    raise ValueError("typefileplot must be 'png' or 'pdf'")

figure, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
figure.patch.set_facecolor("white")
for axis, surface, title in zip(
    axes,
    (relative_prior, relative_likelihood, relative_posterior),
    ("Prior", "Likelihood", "Posterior"),
):
    axis.set_facecolor("white")
    image = axis.pcolormesh(
        log_mu_grid,
        gamma_grid,
        surface,
        shading="auto",
        cmap="magma",
        vmin=0.0,
        vmax=1.0,
    )
    axis.set_xlabel("log10(mu_tilde)")
    axis.set_ylabel("gamma")
    axis.set_title(title)
    axis.grid(False)
    colorbar = figure.colorbar(image, ax=axis)
    colorbar.set_label("Relative density")

visuals_directory = repository_root / "visuals"
visuals_directory.mkdir(exist_ok=True)
figure_path = visuals_directory / f"mass_estimation_single_pair_posterior.{typefileplot}"
print(f"Writing to visuals/{figure_path.name}...")
figure.savefig(
    figure_path,
    dpi=300 if typefileplot == "png" else None,
    bbox_inches="tight",
    facecolor="white",
)
display(figure)
plt.close(figure)

In [ ]:
try:
    from bokeh.models import ColorBar, ColumnDataSource, HoverTool, LinearColorMapper
    from bokeh.palettes import Viridis256
    from bokeh.plotting import figure as bokeh_figure
    from bokeh.transform import transform
    from tdpy.plotting import save_bokeh_html
except ImportError:
    print("Bokeh is not installed; the static figure is available.")
else:
    selected_log_mu = log_mu_grid[::2]
    selected_gamma = gamma_grid[::2]
    selected_posterior = relative_posterior[::2, ::2]
    log_mu_mesh, gamma_mesh = np.meshgrid(selected_log_mu, selected_gamma)
    source = ColumnDataSource({
        "log_mu_tilde": log_mu_mesh.ravel(),
        "gamma": gamma_mesh.ravel(),
        "relative_posterior": selected_posterior.ravel(),
    })
    color_mapper = LinearColorMapper(palette=Viridis256, low=0.0, high=1.0)
    interactive_figure = bokeh_figure(
        title="Single-pair relative posterior grid",
        width=820,
        height=520,
        sizing_mode="stretch_width",
        x_axis_label=r"log10(mu_tilde)",
        y_axis_label="gamma",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    interactive_figure.rect(
        x="log_mu_tilde",
        y="gamma",
        width=float(log_mu_grid[2] - log_mu_grid[0]),
        height=float(gamma_grid[2] - gamma_grid[0]),
        source=source,
        fill_color=transform("relative_posterior", color_mapper),
        line_color=None,
    )
    interactive_figure.add_layout(ColorBar(color_mapper=color_mapper, title="Relative posterior"), "right")
    interactive_figure.add_tools(HoverTool(tooltips=[
        ("log10(mu_tilde)", "@log_mu_tilde{0.00}"),
        ("gamma", "@gamma{0.00}"),
        ("Relative posterior", "@relative_posterior{0.000}"),
    ]))
    interactive_figure.grid.visible = False
    bokeh_path = visuals_directory / "mass_estimation_single_pair_posterior.html"
    print(f"Writing to visuals/{bokeh_path.name}...")
    save_bokeh_html(
        interactive_figure,
        bokeh_path,
        title="Mass-Estimation single-pair posterior",
        typeplotback="white",
    )